In [1]:
import torch
import numpy as np
from transformers import DistilBertModel, DistilBertTokenizer
from torch.utils.data import DataLoader, Dataset
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

CHECKPOINT_PATH = "/content/drive/MyDrive/greenmlops/models/distilbert_baseline.pt"
AG_NEWS_PATH = "/content/drive/MyDrive/greenmlops/src/training/train_distilbert_baseline.py"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Load AG News processed train CSV
df = pd.read_csv("/content/drive/MyDrive/greenmlops/airflow/include/data/processed/ag_news/train.csv")
print(f"AG News train: {df.shape}")
print(df.columns.tolist())
print(df.head(2))

Mounted at /content/drive
Device: cuda
AG News train: (102000, 2)
['text', 'label']
                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2


In [2]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
import numpy as np

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4
)
state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()
print("Model loaded")

def extract_cls_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            max_length=128,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        input_ids = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)
        with torch.no_grad():
            outputs = model.distilbert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_embeddings.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)

# Reference window: first 595 samples
ref_texts = df["text"].iloc[:595].tolist()
ref_embeddings = extract_cls_embeddings(ref_texts)
print(f"Reference embeddings shape: {ref_embeddings.shape}")
print(f"Expected: (595, 768)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded
Reference embeddings shape: (595, 768)
Expected: (595, 768)


In [6]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/greenmlops/src")
from features.embedding_drift import EmbeddingDriftDetector

detector = EmbeddingDriftDetector(rng_seed=42)
fit_info = detector.fit(ref_embeddings)

print("=== Fit diagnostics on real DistilBERT embeddings ===")
for k, v in fit_info.items():
    print(f"  {k}: {v}")

# Clean window - next 256 samples, no drift
clean_texts = df["text"].iloc[595:851].tolist()
clean_embeddings = extract_cls_embeddings(clean_texts)

# Drifted window - 80% topic swap per DRIFT_PROTOCOL.md
# World window receives 80% Sports samples (label 1 receives label 2 samples)
current_texts = df["text"].iloc[595:851].copy()
label_groups = df.groupby("label")["text"].apply(list).to_dict()

# Drifted window: 80% topic swap per DRIFT_PROTOCOL.md
# Take clean_balanced and replace 80% of samples with samples from a different class
n_swap = int(0.8 * len(clean_balanced))  # 204 samples

# Rotation order per protocol: class 0 receives class 1, class 1 receives class 2, etc.
drifted_texts = clean_balanced["text"].tolist().copy()
swap_source = df[df["label"] == 2].iloc[samples_per_class+64:samples_per_class+64+n_swap]["text"].tolist()
drifted_texts[:n_swap] = swap_source

drifted_embeddings_bal = extract_cls_embeddings(drifted_texts)
drifted_result_bal = detector_bal.score(drifted_embeddings_bal)

print("=== Drifted window (80% topic swap) ===")
for k, v in drifted_result_bal.items():
    print(f"  {k}: {v}")

passed = not clean_result_bal["drift_detected"] and drifted_result_bal["drift_detected"]
print(f"\nResult: {'PASS' if passed else 'REVIEW NEEDED'}")

=== Fit diagnostics on real DistilBERT embeddings ===
  pca_explained_variance_ratio: 0.9296
  pca_n_components: 50
  bandwidth: 0.002256
  null_mmd_mean: 0.036226
  null_mmd_std: 0.033883
  mmd_threshold: 0.103991
  n_reference_samples: 595
=== Drifted window (80% topic swap) ===
  drift_score: 0.500375
  drift_detected: True
  mmd_threshold: 0.107512
  ks_flagged: True
  n_components_ks_fail: 36

Result: PASS


In [4]:
print("Reference label distribution:")
print(df["label"].iloc[:595].value_counts().sort_index())
print()
print("Clean window label distribution:")
print(df["label"].iloc[595:851].value_counts().sort_index())

Reference label distribution:
label
0     70
1     58
2     84
3    383
Name: count, dtype: int64

Clean window label distribution:
label
0    95
1    74
2    53
3    34
Name: count, dtype: int64


In [5]:
# Build balanced reference: 148 samples per class = 592 total (~595)
samples_per_class = 148
ref_balanced = pd.concat([
    df[df["label"] == c].iloc[:samples_per_class]
    for c in range(4)
]).reset_index(drop=True)

# Build balanced clean window: 64 samples per class = 256 total
clean_balanced = pd.concat([
    df[df["label"] == c].iloc[samples_per_class:samples_per_class+64]
    for c in range(4)
]).reset_index(drop=True)

print("Balanced reference label distribution:")
print(ref_balanced["label"].value_counts().sort_index())
print()
print("Balanced clean label distribution:")
print(clean_balanced["label"].value_counts().sort_index())

# Re-extract embeddings
ref_embeddings_bal = extract_cls_embeddings(ref_balanced["text"].tolist())
clean_embeddings_bal = extract_cls_embeddings(clean_balanced["text"].tolist())

# Refit detector on balanced reference
detector_bal = EmbeddingDriftDetector(rng_seed=42)
fit_info_bal = detector_bal.fit(ref_embeddings_bal)
print("\n=== Fit diagnostics (balanced) ===")
for k, v in fit_info_bal.items():
    print(f"  {k}: {v}")

# Score clean window
clean_result_bal = detector_bal.score(clean_embeddings_bal)
print("\n=== Clean window (balanced) ===")
for k, v in clean_result_bal.items():
    print(f"  {k}: {v}")

Balanced reference label distribution:
label
0    148
1    148
2    148
3    148
Name: count, dtype: int64

Balanced clean label distribution:
label
0    64
1    64
2    64
3    64
Name: count, dtype: int64

=== Fit diagnostics (balanced) ===
  pca_explained_variance_ratio: 0.9474
  pca_n_components: 50
  bandwidth: 0.001773
  null_mmd_mean: 0.035833
  null_mmd_std: 0.03584
  mmd_threshold: 0.107512
  n_reference_samples: 592

=== Clean window (balanced) ===
  drift_score: 0.029212
  drift_detected: False
  mmd_threshold: 0.107512
  ks_flagged: True
  n_components_ks_fail: 21


In [8]:
# ============================================================
# CIFAR-100 - ResNet-18 embedding drift validation
# ============================================================
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
from torch import nn

RESNET_CHECKPOINT = "/content/drive/MyDrive/greenmlops/models/resnet18_baseline.pt"

model_resnet = torchvision.models.resnet18(weights=None)
model_resnet.fc = nn.Linear(512, 100)
model_resnet.load_state_dict(torch.load(RESNET_CHECKPOINT, map_location=DEVICE))
model_resnet = model_resnet.to(DEVICE)
model_resnet.eval()

embeddings_out = []
def hook_fn(module, input, output):
    embeddings_out.append(output.detach().cpu().numpy())
model_resnet.avgpool.register_forward_hook(hook_fn)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5071, 0.4865, 0.4409],
        std=[0.2673, 0.2564, 0.2762]
    )
])

trainset = torchvision.datasets.CIFAR100(
    root="/content/cifar100", train=True, download=True, transform=transform
)

# Reference window: first 595 samples
ref_subset = torch.utils.data.Subset(trainset, indices=range(595))
ref_loader = torch.utils.data.DataLoader(ref_subset, batch_size=64, shuffle=False)
embeddings_out.clear()
with torch.no_grad():
    for images, _ in ref_loader:
        _ = model_resnet(images.to(DEVICE))
ref_embeddings_resnet = np.concatenate(embeddings_out, axis=0).reshape(-1, 512)
print(f"ResNet reference embeddings: {ref_embeddings_resnet.shape}")

# Fit detector
detector_resnet = EmbeddingDriftDetector(rng_seed=42)
fit_info_resnet = detector_resnet.fit(ref_embeddings_resnet)
print("\n=== ResNet-18 fit diagnostics ===")
for k, v in fit_info_resnet.items():
    print(f"  {k}: {v}")

# Clean window
clean_subset = torch.utils.data.Subset(trainset, indices=range(595, 851))
clean_loader = torch.utils.data.DataLoader(clean_subset, batch_size=64, shuffle=False)
embeddings_out.clear()
with torch.no_grad():
    for images, _ in clean_loader:
        _ = model_resnet(images.to(DEVICE))
clean_embeddings_resnet = np.concatenate(embeddings_out, axis=0).reshape(-1, 512)

# Drifted window - Gaussian noise per DRIFT_PROTOCOL.md
ref_imgs = torch.cat([images for images, _ in ref_loader], dim=0)
per_channel_std = ref_imgs.std(dim=[0, 2, 3])
noise_sigma = 0.5 * per_channel_std.mean().item()

embeddings_out.clear()
with torch.no_grad():
    for images, _ in clean_loader:
        images = images.to(DEVICE)
        noisy = images + torch.randn_like(images) * noise_sigma
        _ = model_resnet(noisy)
drifted_embeddings_resnet = np.concatenate(embeddings_out, axis=0).reshape(-1, 512)

clean_result_resnet = detector_resnet.score(clean_embeddings_resnet)
drifted_result_resnet = detector_resnet.score(drifted_embeddings_resnet)

print("\n=== ResNet-18 clean window ===")
for k, v in clean_result_resnet.items():
    print(f"  {k}: {v}")

print("\n=== ResNet-18 drifted window (0.5 * per-channel std noise) ===")
for k, v in drifted_result_resnet.items():
    print(f"  {k}: {v}")

passed_resnet = (
    not clean_result_resnet["drift_detected"]
    and drifted_result_resnet["drift_detected"]
)
print(f"\nCIFAR-100 Result: {'PASS' if passed_resnet else 'REVIEW NEEDED'}")

100%|██████████| 169M/169M [00:03<00:00, 50.9MB/s]


ResNet reference embeddings: (595, 512)

=== ResNet-18 fit diagnostics ===
  pca_explained_variance_ratio: 0.8416
  pca_n_components: 50
  bandwidth: 0.000774
  null_mmd_mean: 0.047128
  null_mmd_std: 0.014398
  mmd_threshold: 0.075924
  n_reference_samples: 595

=== ResNet-18 clean window ===
  drift_score: 0.054211
  drift_detected: False
  mmd_threshold: 0.075924
  ks_flagged: False
  n_components_ks_fail: 2

=== ResNet-18 drifted window (0.5 * per-channel std noise) ===
  drift_score: 0.435478
  drift_detected: True
  mmd_threshold: 0.075924
  ks_flagged: True
  n_components_ks_fail: 48

CIFAR-100 Result: PASS
